# 🌱 環境変数

VLLM_* と validate_environ。設定すると即死する env を特定。


In [ ]:
import glob
import pathlib
import subprocess
import sys
import zipfile

subprocess.run([sys.executable, "-m", "pip", "download", "--no-cache-dir",
                "--no-deps", "vllm==0.32.0", "-d", "/tmp/w"],
               capture_output=True, text=True)
whl = glob.glob("/tmp/w/*.whl")
print("wheel:", whl)
if not whl:
    raise SystemExit("wheel 取得失敗")
zipfile.ZipFile(whl[0]).extractall("/tmp/vl")
root = pathlib.Path("/tmp/vl/vllm")
print("py files:", len(list(root.rglob("*.py"))))

import os
import re

envs = (root / "envs.py").read_text(errors="replace")
names = sorted(set(re.findall(r'"(VLLM_[A-Z0-9_]+)"', envs)))
print("=== envs.py が読む環境変数:", len(names), "件 ===")

KEY = ("GDN", "ATTENTION_BACKEND", "FLASH", "TRITON", "USE_V1", "COMPILE",
       "CAPTURE", "WORKER", "MAX_JOBS", "CACHE", "TORCH", "DEVICE", "CUDA",
       "MULTIPROC", "SLEEP", "KV", "NIXL", "S3", "LOGGING", "VERBOSE")
hits = [n for n in names if any(k in n for k in KEY)]
for n in hits:
    m = re.search(r'"' + n + r'"\s*:\s*lambda[^\n]*', envs)
    line = m.group(0) if m else ""
    cur = os.environ.get(n)
    print(f"  {n:44s} set={cur!r}  {line[:90]}")

print("\n=== validate_environ（即死する env）===")
for f in ("platforms/cuda.py", "platforms/interface.py"):
    t = (root / f).read_text(errors="replace")
    i = t.find("def validate_environ")
    if i < 0:
        continue
    print(f"--- {f} ---")
    print(t[i:i + 2200])

print("\n=== capability で分岐するフラグ候補 ===")
for f in ("platforms/cuda.py", "platforms/interface.py",
          "v1/attention/backends/registry.py"):
    t = (root / f).read_text(errors="replace")
    for i, l in enumerate(t.splitlines(), 1):
        if "has_device_capability" in l or "is_device_capability" in l:
            print(f"  {f}:{i}: {l.strip()[:150]}")

print("\n=== 実行環境の env（VLLM_/CUDA_/TORCH_ のみ）===")
for k in sorted(os.environ):
    if k.startswith(("VLLM_", "CUDA_", "TORCH_", "NCCL_")):
        print(f"  {k}={os.environ[k][:80]}")
